# Esquema: regresión lineal (MVP manual)

Versión **mínima viable**: CSV → pandas → **tratamiento manual** → split → **varios `Pipeline`** (mismo preprocesado manual, distinto modelo).

> En [07.b](../07.b-ejemplos-supervisados/) el imputer/encoder van **dentro** del `Pipeline`. Aquí el escalado+modelo sí van en pipeline; la limpieza va **a mano** en pandas.

| Paso | Qué hace |
|------|----------|
| 1–4 | CSV, target numérico, `fillna`, `get_dummies` |
| 5 | Split train / val / test |
| 6 | Entrenar regresores en `Pipeline` |
| 7 | Análisis comparativo (val → test) |

Siguiente: [07.b regresión](../07.b-ejemplos-supervisados/01-regresion-lineal.ipynb).

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


## 1. Importar CSV y revisar datos

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("data/datos_casas.csv")
print("Tipos:\n", df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Tipos:
 Metros_Cuadrados    float64
Zona                 object
Precio                int64
dtype: object

Faltantes:
 Metros_Cuadrados    1
Zona                1
Precio              0
dtype: int64


,Metros_Cuadrados,Zona,Precio
0,45.0,Centro,125000
1,52.0,Periferia,138000
2,NaN,Centro,132000
3,61.0,Periferia,155000
4,70.0,Centro,172000
5,85.0,NaN,195000
6,95.0,Periferia,210000
7,110.0,Centro,245000
8,130.0,Periferia,285000
9,150.0,Centro,320000


## 2. Target numérico (`Precio`)

In [26]:
df = df.dropna(subset=["Precio"]).copy()
y = df["Precio"]
print("Filas:", len(df))


Filas: 11


## 3–4. Features (manual)

In [27]:
metros = df["Metros_Cuadrados"].astype(float)
X_num = pd.DataFrame({"Metros_Cuadrados": metros.fillna(metros.median())})

zona = df["Zona"].fillna("Desconocida").astype(str)
X_cat = pd.get_dummies(zona, prefix="Zona", dtype=float)

X = pd.concat([X_num, X_cat], axis=1)
display(X.head())


,Metros_Cuadrados,Zona_Centro,Zona_Desconocida,Zona_Periferia
0,45.0,1.0,0.0,0.0
1,52.0,0.0,0.0,1.0
2,90.0,1.0,0.0,0.0
3,61.0,0.0,0.0,1.0
4,70.0,1.0,0.0,0.0


## 5. Split train / val / test


In [28]:
def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final).
    - val_size: fracción de train+val → validación.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test


RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"Tamaños → train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")


Tamaños → train: 6 | val: 2 | test: 3


## 6. Entrenar modelos en Pipeline

Bucle sobre `build_models()`: `make_pipeline(StandardScaler(), modelo)`, **fit** solo en train. Los modelos ajustados se guardan en `pipelines` para el análisis del paso 7.

Solo **`fit` en train**; predicciones y métricas van en el apartado de análisis.


In [29]:
def build_models():
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "DecisionTree": DecisionTreeRegressor(
            criterion="squared_error",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            criterion="squared_error",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=1.0,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()


pipelines = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe


## 7. Análisis comparativo

Tabla ordenada por **R² en val**; el ganador se evalúa en **test**.

Predicciones en **val** y **test**, tabla comparativa y elección del ganador por **val**.


In [30]:
filas = []
for nombre, pipe in pipelines.items():
    pred_val = pipe.predict(X_val)
    pred_test = pipe.predict(X_test)
    filas.append(
        {
            "modelo": nombre,
            "R2_val": r2_score(y_val, pred_val),
            "R2_test": r2_score(y_test, pred_test),
            "MSE_val": mean_squared_error(y_val, pred_val),
            "MSE_test": mean_squared_error(y_test, pred_test),
        }
    )

tabla = pd.DataFrame(filas).sort_values("R2_val", ascending=False)
display(tabla.round(4))

mejor = tabla.iloc[0]
print(f"\nMejor R² en val: {mejor['modelo']} ({mejor['R2_val']:.4f})")
print(f"R² en test del ganador: {mejor['R2_test']:.4f}")


,modelo,R2_val,R2_test,MSE_val,MSE_test
8,XGBoost,0.5575,0.9015,1.412500e+09,6.410009e+08
4,RandomForest,0.3177,0.9371,2.178004e+09,4.092455e+08
1,Ridge,0.1630,0.9776,2.671839e+09,1.455775e+08
2,Lasso,0.0474,0.9987,3.040866e+09,8.391811e+06
0,LinearRegression,0.0473,0.9983,3.041103e+09,1.113918e+07
7,KNN,-0.0038,0.4449,3.204500e+09,3.611387e+09
9,CatBoost,-0.0310,0.2905,3.291363e+09,4.615718e+09
5,GradientBoosting,-0.0359,0.9158,3.306905e+09,5.480485e+08
3,DecisionTree,-0.1448,0.8125,3.654500e+09,1.219667e+09
6,HistGradientBoosting,-0.3801,-0.0154,4.405611e+09,6.605556e+09



Mejor R² en val: XGBoost (0.5575)
R² en test del ganador: 0.9015
